# 23 · Context Engineering：交给 LLM 之前

> 检索 ≠ 全塞。**召回的 10 个 chunk 直接全扔给 LLM 是最常见的偷懒**。Context Engineering 负责在“检索结果”与“最终 Prompt”之间加一层精加工。

**本文件覆盖知识点**：Context Selection / Compression / Filtering / Deduplication / Ordering / Truncation

```text
Retrieval → Rerank → 去重 → 过滤 → 压缩 → 排序 → LLM
```

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


In [ ]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集.md 是「人工标注的答案」，不能进索引 —— 否则第 33 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里显式排除。
_EXCLUDE = {'评测集.md'}

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name in _EXCLUDE:
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


## 1. 为什么不能“全部塞给 LLM”

1. **噪声干扰**：不相关的片段会“带偏”模型（尤其夹在开头附近）；
2. **成本**：token 即金钱，塞得越多越贵；
3. **效果**：研究 Lost-in-the-Middle 显示，被埋在中间的信息利用率差。

所以上线前要过这几道“闸”：

## 2. 五道闸

| 闸 | 做什么 | 手段 |
|----|--------|------|
| **去重 Dedup** | 同一事实被多篇文档复述，去重 | 文本/语义相似度聚类 |
| **过滤 Filter** | 丢掉阈值以下/类别不对的 | 相似度阈值、元数据、LLM 判官 |
| **压缩 Compress** | 长 chunk 掐头去尾留精华 | LLM 摘要、句子级抽取 |
| **排序 Order** | 把最相关的放最前/最后 | 按重排分降序 |
| **截断 Truncate** | 超出预算就砍 | 按 token 预算逐段放入 |

In [ ]:
# 知识点·真调说明：过滤 / 压缩 —— 五道闸里“LLM判官过滤 + LLM摘要压缩”长什么样
# 候选片段不再是手写的假文档，而是底座在真实语料上召回的片段。
import json as _json
flt_q = '星云智能客服标准版怎么收费？'
flt_hits = hybrid_retrieve(flt_q, k=4)                 # 真实混合检索 Top-4 当候选
flt_docs = [h['text'] for h in flt_hits]
print('① LLM 判官过滤 —— 真实召回 %d 条，只保留“能直接回答收费问题”的:' % len(flt_docs))
for i, h in enumerate(flt_hits, 1):
    print('   [%d] %s · %s :: %s' % (i, h['source'], h['section'], h['text'][:34].replace('\n', ' ')))
flt_text = '\n'.join('[%s] %s' % (i + 1, t) for i, t in enumerate(flt_docs))
out1 = _llm_live(
    prompt='问题：%s\n\n候选片段：\n%s\n\n请逐条判断哪些片段与问题相关、值得保留给后续生成。' % (flt_q, flt_text),
    system='你是 RAG 的“上下文过滤判官”。规则：只保留能直接帮助回答该问题的片段；'
           '沾边但无助于本题、或讲其它版本/无关话题的一律剔除。只输出 JSON，禁止其它文字：'
           '{"keep_ids": [保留的编号], "drop_reasons": {"被剔除编号": "一句话原因"}}。',
    fallback='未配置 Key 的固定样例：\n'
             '{"keep_ids": [1], "drop_reasons": {"2": "介绍产品定位与技术原理，未提及标准版收费", '
             '"3": "仅说明文档内容范围，未给出具体收费标准", "4": "描述产品场景、能力和历史，与收费无关"}}',
    temperature=0.1,
)
if out1 is None:
    out1 = ('{"keep_ids": [1], "drop_reasons": {"2": "介绍产品定位与技术原理，未提及标准版收费", '
            '"3": "仅说明文档内容范围，未给出具体收费标准", "4": "描述产品场景、能力和历史，与收费无关"}}')
    print('（以上为固定样例；下面用样例演示 JSON 解析）')
try:
    o1 = _json.loads(out1[out1.find('{'): out1.rfind('}') + 1])
    print('保留:', o1['keep_ids'], ' 剔除:', o1.get('drop_reasons'))
except Exception as e:
    print('JSON 解析失败：', e, '—— 说明要收紧输出约束。')

print()
print('② LLM 摘要压缩 —— 把真实召回里最长的一段“掐头去尾”留能回答问题的要点')
long_doc = max(flt_docs, key=len)
print('   原始片段 %d 字：%s' % (len(long_doc), long_doc[:60].replace('\n', ' ')))
out2 = _llm_live(
    prompt='请把下面这段资料压缩成不超过 3 句的精炼版，只保留回答“%s”所需的要点，'
           '删掉与问题无关的内容与修饰。\n资料：%s' % (flt_q, long_doc),
    system='你是 RAG 上下文压缩器。输出必须是资料里真实存在的信息，不要新增内容。',
    fallback='未配置 Key 的固定样例：\n'
             '星云智能客服标准版按版本订阅，收费为 998 元/月。\n'
             '公有云版本按成功返回的对话轮次计费，失败请求不计费；该版本不占用坐席数。',
    temperature=0.2,
)
if out2 is None:
    out2 = ('星云智能客服标准版按版本订阅，收费为 998 元/月。\n'
            '公有云版本按成功返回的对话轮次计费，失败请求不计费；该版本不占用坐席数。')
    print('（以上为固定样例；以下用样例演示长度对比）')
print('原文 %d 字 → 压缩后 %d 字（送入 LLM 的 token 随之减少，成本下降）' % (len(long_doc), len(out2)))
print('→ 过滤丢“看似相关实则无关”的噪声、压缩省 token——这是生成前替 LLM 把关的两道闸。')


In [ ]:
# 真实上下文构建：真实检索 + 真实重排 → 语义去重 → token 预算截断 → 带来源标注拼装 → 生成
# 这一层存在的意义：召回的片段不能“一把梭”全塞给 LLM（噪声干扰 / token 成本 / Lost-in-the-Middle）。
import numpy as np

CTX_QUERY = '各个版本的价格分别是多少？'
DUP_THRESHOLD = 0.88    # 片段间余弦超过它判为「同一事实被多篇文档复述」；0.88 是本语料实测的合理分界
BUDGET_TOKENS = 600     # 近似 token 预算（中文 1 字≈1 token，英文/数字 4 字符≈1 token）

def est_tokens(t):
    """近似 token 数：中文按 1 字 1 token、其余按 4 字符 1 token（生产环境应换成真 tokenizer）"""
    zh = sum(1 for ch in t if u'\u4e00' <= ch <= u'\u9fff')
    return int(zh + (len(t) - zh) / 4 + 0.5)

def build_context(query, pool=10, top_n=8, budget=BUDGET_TOKENS, dup_thr=DUP_THRESHOLD):
    """把检索结果精加工成喂给 LLM 的上下文：去重 → 过滤 → 排序 → 截断 → 拼装"""
    # ① 召回：底座真实混合检索（FAISS 向量 + BM25），先取一个偏大的候选池
    cands = hybrid_retrieve(query, k=pool)
    print('① 真实混合检索候选池 %d 条:' % len(cands))
    for i, c in enumerate(cands, 1):
        print('   %2d. [%s · %s] %s' % (i, c['source'], c['section'], c['text'][:34].replace('\n', ' ')))

    # ② 精排：真调 qwen3-rerank 拿真实相关性分数，顺带完成「过滤」（只留 Top-n）
    if _HAS_KEY:
        ranked = rerank(query, [c['text'] for c in cands], top_n=top_n)
        by_text = {c['text']: c for c in cands}
        docs = [dict(by_text[t], score=s) for t, s in ranked]
    else:
        docs = [dict(c, score=1.0 - 0.05 * i) for i, c in enumerate(cands[:top_n])]
        print('   （无 Key：重排是现场调用，这里退回检索名次当近似分数）')
    print('\n② 精排后保留 Top-%d（%s，降序）:'
          % (len(docs), '真实 relevance 分' if _HAS_KEY else '检索名次近似分(无 Key)'))
    for d in docs:
        print('   %.6f  [%s · %s] %s' % (d['score'], d['source'], d['section'], d['text'][:36].replace('\n', ' ')))

    # ③ 语义去重：真调 embed 算片段两两余弦，超过阈值判为「同一事实」，只留分高的那条
    #    （比「开头 N 字相同才去重」强：换种说法复述同一张价目表也能识别出来）
    vecs = embed([d['text'] for d in docs])
    keep, dups = [], []
    for i, d in enumerate(docs):
        hit = None
        for j in keep:
            cos = float(vecs[i] @ vecs[j])
            if cos > dup_thr:
                hit = (j, cos)
                break
        if hit:
            dups.append((i, hit[0], hit[1]))
        else:
            keep.append(i)
    print('\n③ 语义去重（embed 余弦 > %.2f 判为同一事实复述）: 命中 %d 组重复，丢弃 %d 条'
          % (dup_thr, len(dups), len(dups)))
    for i, j, cos in dups:
        print('   丢 [%s · %s]（与 [%s · %s] 余弦 %.4f，同一事实只保留分高的一条）'
              % (docs[i]['source'], docs[i]['section'], docs[j]['source'], docs[j]['section'], cos))
    below = [(float(vecs[a] @ vecs[b]), a, b) for a in range(len(docs)) for b in range(a + 1, len(docs))]
    below = [x for x in below if x[0] <= dup_thr]
    if below:
        c0, a0, b0 = max(below)
        print('   最接近阈值的未命中对: %.4f（[%s] vs [%s]）—— 在阈值之下，没有被误杀'
              % (c0, docs[a0]['section'], docs[b0]['section']))

    # ④ 截断：按 token 预算逐段放入（相关度高的先放），放不下的直接丢尾
    kept, used, cut = [], 0, []
    for i in sorted(keep, key=lambda x: -docs[x]['score']):
        t = est_tokens(docs[i]['text'])
        if used + t > budget:
            cut.append((i, t))
            continue
        used += t
        kept.append((docs[i], t))
    print('\n④ token 预算截断（预算 %d tok，按相关度高的优先放）:' % budget)
    acc = 0
    for n, (d, t) in enumerate(kept, 1):
        acc += t
        print('   留 片段%d  %4d tok（累计 %4d）  [%s · %s]' % (n, t, acc, d['source'], d['section']))
    for i, t in cut:
        print('   丢 [%s · %s]  需 %d tok，超出预算' % (docs[i]['source'], docs[i]['section'], t))
    print('   实际占用 %d / %d tok' % (used, budget))

    # ⑤ 拼装：按分数降序（最相关的放最前），每段带来源标注，生成后可据此溯源
    parts = ['[片段%d | 来源《%s》 | 章节：%s | 相关度 %.4f]\n%s'
             % (n, d['source'], d['section'], d['score'], d['text'])
             for n, (d, _) in enumerate(kept, 1)]
    ctx = '\n\n'.join(parts)
    print('\n⑤ 最终上下文（%d 段 / %d tok）:' % (len(kept), used))
    print(ctx)
    return ctx, [d for d, _ in kept]

context, kept_docs = build_context(CTX_QUERY)

# ⑥ 把拼好的上下文喂给 qwen-plus —— 这才是这一步的产出物
print('\n⑥ 把上面拼好的上下文喂给 qwen-plus 生成答案:')
_ans = chat('仅依据下面的资料回答问题，资料里没有的就直说不知道；不要用资料外的知识补充。\n\n'
            '资料：\n%s\n\n问题：%s' % (context, CTX_QUERY))
if _ans is None:
    recorded("""根据资料，各个版本的价格如下：

- 基础版：298 元/月
- 标准版：998 元/月
- 专业版：2980 元/月
- 企业版：按需报价

以上信息来源于片段1和片段2。""", '录制于 2026-09-12，模型 qwen-plus')
else:
    print(_ans)


In [ ]:
# 知识点·真调说明：去重 —— 开头50字粗去重拦不住“换了说法”的重复，语义判官才行
# 对比用的片段对来自真实召回：价目表在《计费与SLA》与《产品手册》里被复述，就是真实的重复。
import json as _json
b_hits = hybrid_retrieve('各个版本的价格分别是多少？', k=4)
b_pairs = [(b_hits[0]['text'], b_hits[1]['text']), (b_hits[2]['text'], b_hits[3]['text'])]
print('用作对比的真实片段对:')
for i, (a, b) in enumerate(b_pairs, 1):
    ma, mb = b_hits[2 * (i - 1)], b_hits[2 * (i - 1) + 1]
    print('  第%d对 A《%s》·%s: %s' % (i, ma['source'], ma['section'], a[:36].replace('\n', ' ')))
    print('        B《%s》·%s: %s' % (mb['source'], mb['section'], b[:36].replace('\n', ' ')))
print()
print('上面朴素 builder 的开头 50 字去重：')
for i, (a, b) in enumerate(b_pairs, 1):
    print('  第%d对 开头50字是否相同=%s（相同才可能被它去重）' % (i, a[:50] == b[:50]))
print()
b_prompt = '\n'.join('第%d对：\nA. %s\nB. %s' % (i + 1, a, b) for i, (a, b) in enumerate(b_pairs))
out = _llm_live(
    prompt='请判断下面每一对片段是否在“陈述同一个事实（属于重复信息）”。\n%s' % b_prompt,
    system='你是 RAG 的语义去重判官：两段只要说的是同一件事就标 true（哪怕措辞完全不同），'
           '信息互补/各说各的就标 false。只输出 JSON：'
           '{"pairs": [{"duplicate": true或false, "reason": "一句话"}]}。禁止其它文字。',
    fallback='未配置 Key 的固定样例：\n'
             '{"pairs": [{"duplicate": true, "reason": "两段均完整列出了基础版、标准版、专业版、企业版的版本名称、'
             '价格、坐席数、知识库容量和并发上限，核心表格内容完全一致，属于同一事实的重复陈述。"}, '
             '{"duplicate": false, "reason": "A段是关于计费规则的FAQ问答，涵盖收费方式、坐席定义、超限处理和退款政策；'
             'B段是代码片段及展示建议，涉及API调用和references输出，二者主题、信息类型与内容无重叠。"}]}',
    temperature=0.1,
)
if out is None:
    out = ('{"pairs": [{"duplicate": true, "reason": "两段均完整列出了基础版、标准版、专业版、企业版的版本名称、'
           '价格、坐席数、知识库容量和并发上限，核心表格内容完全一致，属于同一事实的重复陈述。"}, '
           '{"duplicate": false, "reason": "A段是关于计费规则的FAQ问答，涵盖收费方式、坐席定义、超限处理和退款政策；'
           'B段是代码片段及展示建议，涉及API调用和references输出，二者主题、信息类型与内容无重叠。"}]}')
    print('（以上为固定样例；下面用样例演示 JSON 解析）')
try:
    objs = _json.loads(out[out.find('{'): out.rfind('}') + 1])['pairs']
    for i, o in enumerate(objs, 1):
        verdict = '重复信息，应去重' if o['duplicate'] else '信息互补，两个都保留'
        print('  第%d对 → %s（%s）' % (i, verdict, o.get('reason', '')))
    print('→ 换了说法的语义重复，开头截断式哈希抓不到；这正是上面 build_context 里用 embed 余弦去重的原因。')
except Exception as e:
    print('JSON 解析失败：', e, '—— 说明输出约束不够严。')


## 3. Context 的顺序怎么放

- 相关片段**放开头**最容易让模型注意到；
- 全放开头有时反而浪费——长文档场景试“最重要的在首尾，次要在中间”。

进阶上下文技术（**给片段补上下文、父子块、句窗**）见下一课（24）。

## 小结

- 别把召回结果“一把梭”喂给 LLM；
- 五道闸：**去重→过滤→压缩→排序→截断**；
- 上下文质量直接决定生成质量，也直接关系 token 成本。